# 13. 학습 이미지 수 비교: 0 / 5 / 10 / 전체

직접 나눈 train/validation/test 폴더를 사용합니다. 0장은 공식 SAM2.1 단일 마스크 기준선이며 optimizer update는 없습니다. 각 학습은 공식 가중치에서 새로 시작합니다. Validation으로 선택한 best checkpoint를 학습 종료 직후 test하여 표와 마스크를 출력합니다.

공식 모델 모듈을 사용하는 자체 decoder-only 학습 루프입니다. 공식 MOSE Trainer가 아닙니다. 기존 TEM_Offline_Metrology 패키지의 setup·vendor·공식 SAM2.1 Large 가중치가 필요합니다. DINO는 학습하지 않습니다.

5장 ⊂ 10장 ⊂ 전체로 구성합니다. 선택한 클래스가 포함되도록 먼저 이미지를 고른 뒤 나머지를 고정 seed로 섞습니다. 따라서 완전 무작위 표본은 아니며, 사용 이미지 ID를 저장합니다. Validation/test는 모든 실험에서 고정합니다. Test 결과로 설정을 반복 조정하지 마세요.

In [ ]:
from pathlib import Path
import sys, json, time, copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
ROOT = Path('/content/TEM_Offline_Metrology') if Path('/content/TEM_Offline_Metrology').exists() else Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
# ROOT = Path('/company/path/TEM_Offline_Metrology')
assert (ROOT/'temflow').is_dir(), f'패키지 ROOT를 수정하세요: {ROOT}'
assert (ROOT/'weights/sam2.1_hiera_large.pt').is_file(), 'SAM 가중치가 필요합니다'
sys.path.insert(0,str(ROOT))
from temflow.models import Models
from temflow.data import inspect_gt, preview_dataset
from temflow.core import read_image, digest
DATA_ROOT = ROOT/'semantic_training'
# DATA_ROOT = Path('/company/datasets/tem_layers')
# 아래 train / validation / test 폴더는 직접 준비합니다. 자동 분할하지 않습니다.
print('Model root:',ROOT,'Data root:',DATA_ROOT)


## 1. 직접 분리한 폴더 경로
다음 구조로 원본과 GT를 준비합니다. 원본과 GT는 확장자를 제외한 파일명이 같아야 합니다. 폴더 내부의 파일을 읽으며 하위 폴더는 자동 탐색하지 않습니다.
```text
DATA_ROOT/
  train/images/         train/gt/
  validation/images/    validation/gt/
  test/images/          test/gt/
```
아래 6개 경로를 각각 다른 위치로 지정해도 됩니다. 모든 split은 비어 있지 않아야 합니다. 자동 랜덤 분할은 없습니다. 원본 uint8 이미지, 정수 ID 또는 정확한 RGB GT PNG/TIFF를 사용합니다. 정규화 GT 사본과 manifest는 패키지 아래 별도 폴더에 저장하며 입력 파일은 변경하지 않습니다.

In [ ]:
SPLIT_DIRS = {
    'train': {'images': DATA_ROOT/'train/images', 'gt': DATA_ROOT/'train/gt'},
    'val':   {'images': DATA_ROOT/'validation/images', 'gt': DATA_ROOT/'validation/gt'},
    'test':  {'images': DATA_ROOT/'test/images', 'gt': DATA_ROOT/'test/gt'},
}
for split, paths in SPLIT_DIRS.items():
    print(split, paths)
    inspect_gt(paths['gt'])


## 2. 클래스 정의와 폴더 검사
CLASS_NAMES, TARGET_CLASSES, RGB_MAP을 실제 GT에 맞게 수정하세요. 팔레트 PNG는 저장된 인덱스를 사용합니다. 동일 픽셀의 이미지가 split 사이에 중복되면 중단합니다. 이름만 같은 서로 다른 이미지는 허용합니다.

동일 시편의 다른 이미지까지 자동으로 판별할 수는 없습니다. 필요하면 GROUPS에 'train/image01': 'wafer_A'처럼 시편명을 지정하세요. 같은 그룹이 서로 다른 split에 있으면 오류를 표시합니다. train/validation 각각 선택한 모든 클래스가 필요하며 test는 실제 있는 클래스만 평가합니다.

In [ ]:
"""Read user-defined splits without moving or automatically splitting inputs."""
from pathlib import Path
import hashlib, json
import numpy as np
from PIL import Image
from temflow.core import read_image, digest
from temflow.data import decode_gt

def load_folder_splits(split_dirs, prepared_dir, class_names, rgb_map=None,
                       ignore=255, groups=None):
    groups=groups or {}; prepared_dir=Path(prepared_dir).resolve()
    if prepared_dir.exists() and any(prepared_dir.iterdir()):
        raise ValueError('Use a new empty prepared_dir to preserve previous manifests')
    if set(split_dirs)!={'train','val','test'}:
        raise ValueError('Configure train, val and test paths explicitly')
    rows=[]; arrays=[]; pixel_splits={}; group_splits={}
    def index(folder,extensions):
        folder=Path(folder).expanduser().resolve()
        if not folder.is_dir():raise ValueError('Missing folder: '+str(folder))
        files={}
        for p in sorted(folder.iterdir()):
            if not p.is_file() or p.suffix.lower() not in extensions:continue
            key=p.stem.casefold()
            if key in files:raise ValueError('Duplicate stem: '+str(p))
            files[key]=p
        if not files:raise ValueError('Empty input folder: '+str(folder))
        return files
    for split,dirs in split_dirs.items():
        images=index(dirs['images'],{'.png','.tif','.tiff','.jpg','.jpeg'})
        gts=index(dirs['gt'],{'.png','.tif','.tiff'})
        if set(images)!=set(gts):
            raise ValueError(f'{split}: missing GT {sorted(set(images)-set(gts))}; unmatched GT {sorted(set(gts)-set(images))}')
        for key,p in images.items():
            gt=gts[key]; im=read_image(p)
            with Image.open(gt) as src:
                a=np.asarray(src)
                if getattr(src,'n_frames',1)!=1:raise ValueError('Multi-page GT unsupported: '+str(gt))
                if a.ndim==3 and a.shape[2]==4 and np.any(a[:,:,3]!=255):
                    raise ValueError('Transparent GT must be explicitly converted to ignore')
                if a.ndim==2 and a.dtype.kind not in 'uib':raise ValueError('GT IDs must be integers')
            a=decode_gt(gt,rgb_map=rgb_map)
            if a.shape!=im.shape[:2]:raise ValueError('Image/GT size mismatch: '+str(p))
            unknown=set(map(int,np.unique(a)))-set(map(int,class_names))-{ignore}
            if unknown:raise ValueError(f'{p.name}: undefined classes {unknown}')
            ph=hashlib.sha256(str(im.shape).encode()+im.tobytes()).hexdigest()
            if ph in pixel_splits and pixel_splits[ph]!=split:
                raise ValueError('Identical image pixels across splits: '+str(p))
            pixel_splits[ph]=split
            ident=f'{split}/{p.stem}'
            group=groups.get(ident,ident)
            if group in group_splits and group_splits[group]!=split:
                raise ValueError('Same specimen group across splits: '+str(group))
            group_splits[group]=split
            dest=prepared_dir/split/(p.stem+'.png')
            rows.append(dict(id=ident,image=str(p),gt=str(dest),source_gt=str(gt),
                source_gt_sha256=digest(gt),group=group,split=split,image_sha256=digest(p),
                pixel_sha256=ph,classes=[int(c) for c in np.unique(a) if c!=ignore]))
            arrays.append(a.astype('uint16'))
    # Write only normalized copies, after all source data has passed validation.
    for r,a in zip(rows,arrays):
        dest=Path(r['gt']);dest.parent.mkdir(parents=True,exist_ok=True)
        Image.fromarray(a).save(dest);r['gt_sha256']=digest(dest)
    (prepared_dir/'manifest.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf8')
    return rows


In [ ]:
CLASS_NAMES = {0:'background',1:'upper_layer',2:'middle_layer',3:'lower_layer',255:'ignore'}
TARGET_CLASSES = [1,2,3]
IGNORE_ID = 255
RGB_MAP = None  # 예: {'0,0,0':0,'255,0,0':1,'0,255,0':2,'0,0,255':3,'255,255,255':255}
GROUPS = {}  # 예: {'train/image01':'wafer_A', 'train/image02':'wafer_A'}
import uuid
PREPARED = ROOT/'semantic_training_prepared'/uuid.uuid4().hex
rows = load_folder_splits(SPLIT_DIRS,PREPARED,CLASS_NAMES,rgb_map=RGB_MAP,ignore=IGNORE_ID,groups=GROUPS)
assert set(TARGET_CLASSES)<=set(CLASS_NAMES) and not ({0,IGNORE_ID}&set(TARGET_CLASSES))
preview_dataset(DATA_ROOT,rows)
plt.show()
print('Normalized GT + manifest:',PREPARED)
print([(r['id'],r['split'],r['classes']) for r in rows])


## 3. 학습 함수
아래 셀은 노트북에 포함된 학습 구현입니다. 이미지·GT hash 및 split을 검사하고, 클래스 균등 샘플링, 4개 crop 특징 캐시, ignore 제외 손실, 고정 검증 프롬프트를 사용합니다. GT 연결 성분을 하나의 목표로 취급하므로 붙어 있는 같은 클래스 객체는 별도 인스턴스/ROI 준비가 필요합니다.

단일 출력 경로를 학습하므로 학습 가중치 적용 후에도 single-mask 추론을 사용합니다. 기본 멀티마스크 모델과 결과를 비교할 때 프롬프트·출력 선택 규칙을 통일하세요.

In [ ]:
"""Multi-class GT -> prompted binary masks; custom SAM2.1 decoder-only loop.

Not Meta's official Trainer. Uses its model modules with BCE/Dice/quality losses.
"""
import json, random, time
from pathlib import Path
from collections import OrderedDict
import numpy as np
from PIL import Image
from scipy.ndimage import label, distance_transform_edt
from temflow.core import read_image, digest


def make_samples(root, rows, classes, ignore=255, pad=64, min_area=8):
    samples=[]
    for r in rows:
        a=np.asarray(Image.open(Path(root)/r['gt']))
        if a.ndim!=2: raise ValueError('Normalized integer GT required')
        for cls in classes:
            cc,n=label(a==cls)
            for k in range(1,n+1):
                yy,xx=np.where(cc==k)
                if len(xx)<min_area: continue
                x0=max(0,int(xx.min())-pad); x1=min(a.shape[1],int(xx.max())+pad+1)
                y0=max(0,int(yy.min())-pad); y1=min(a.shape[0],int(yy.max())+pad+1)
                target=cc[y0:y1,x0:x1]==k
                if target.shape[0]<2 or target.shape[1]<2: continue
                samples.append(dict(row=r,cls=int(cls),crop=[x0,y0,x1,y1],target=target,
                                    valid=a[y0:y1,x0:x1]!=ignore))
    return samples


def prompt(sample, rng, mode='box_points', jitter=False):
    t=sample['target']; valid=sample['valid']; h,w=t.shape
    yy,xx=np.where(t); b=np.array([xx.min(),yy.min(),xx.max(),yy.max()],float)
    if jitter: b+=rng.integers(-2,3,4)
    b=np.clip(b,[0,0,0,0],[w-1,h-1,w-1,h-1])
    b[2]=min(w-1,max(b[0]+1,b[2]));b[0]=min(b[0],b[2]-1)
    b[3]=min(h-1,max(b[1]+1,b[3]));b[1]=min(b[1],b[3]-1)
    pts=[]; labs=[]
    if mode!='box':
        if jitter:
            ids=rng.choice(len(xx),size=int(rng.choice([1,3])),replace=True)
            pts=[[float(xx[i]),float(yy[i])] for i in ids]
        else:
            y,x=np.unravel_index(distance_transform_edt(t).argmax(),t.shape)
            pts=[[float(x),float(y)]]
        labs=[1]*len(pts)
        # Deterministic negatives during validation; sample only valid non-target pixels.
        near=(~t)&valid&(distance_transform_edt(~t)<=12)
        ny,nx=np.where(near)
        if len(nx):
            ids=rng.choice(len(nx),size=min(3,len(nx)),replace=False) if jitter else np.linspace(0,len(nx)-1,min(3,len(nx)),dtype=int)
            pts.extend([[float(nx[i]),float(ny[i])] for i in ids]);labs.extend([0]*len(ids))
    return np.array(pts,float).reshape(-1,2),np.array(labs,int),None if mode=='points' else b


def train_semantic(models,root,rows,classes,out,steps=200,lr=1e-5,pad=64,
                   min_area=8,ignore=255,seed=42,validate_every=20,
                   modes=('box_points','points','box'),quality_weight=.1):
    import torch
    import torch.nn.functional as F
    root=Path(root);out=Path(out)
    if (out/'metadata.json').exists(): raise ValueError('Use a new run output directory')
    classes=sorted(set(map(int,classes)))
    if not classes or 0 in classes or ignore in classes: raise ValueError('Choose foreground IDs')
    if steps<0 or validate_every<1 or lr<=0: raise ValueError('Invalid training settings')
    if not modes or set(modes)-{'box_points','points','box'}: raise ValueError('Invalid prompt modes')
    splits={s:{r['group'] for r in rows if r['split']==s} for s in ('train','val','test')}
    if any(splits[a]&splits[b] for a,b in [('train','val'),('train','test'),('val','test')]):
        raise ValueError('Group leakage across splits')
    hashes={}
    for r in rows:
        if r['split'] not in splits: raise ValueError('Unknown split')
        if digest(root/r['image'])!=r['image_sha256'] or digest(root/r['gt'])!=r['gt_sha256']:
            raise ValueError('Data changed after preparation')
        ph=r.get('pixel_sha256',r['image_sha256'])
        if ph in hashes and hashes[ph]!=r['split']: raise ValueError('Duplicate image across splits')
        hashes[ph]=r['split']
    sets={s:make_samples(root,[r for r in rows if r['split']==s],classes,ignore,pad,min_area) for s in splits}
    for s in ('train','val'):
        missing=set(classes)-{v['cls'] for v in sets[s]}
        if missing: raise ValueError(f'{s} missing classes: {missing}')
    out.mkdir(parents=True,exist_ok=True)
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);rng=np.random.default_rng(seed)
    model=models.sam.model;model.eval();device=models.device
    for p in model.parameters():p.requires_grad_(False)
    # Single-mask path plus quality head. Other output MLPs/object head unused.
    for n,p in model.sam_mask_decoder.named_parameters():
        if n.startswith('pred_obj_score_head'):continue
        if n.startswith('output_hypernetworks_mlps.') and not n.startswith('output_hypernetworks_mlps.0.'):continue
        p.requires_grad_(True)
    named={n:p for n,p in model.named_parameters() if p.requires_grad}
    assert named and all(n.startswith('sam_mask_decoder.') for n in named)
    initial={n:p.detach().cpu().clone() for n,p in named.items()}
    dynamic=model.sam_mask_decoder.dynamic_multimask_via_stability
    model.sam_mask_decoder.dynamic_multimask_via_stability=False
    opt=torch.optim.AdamW(named.values(),lr=lr,weight_decay=.01)
    cache=OrderedDict(); timing=dict(feature_seconds=0.,train_seconds=0.,eval_seconds=0.)
    def sync():
        if str(device).startswith('cuda'):torch.cuda.synchronize()
    def forward(s,mode,jitter):
        r=s['row'];x0,y0,x1,y1=s['crop'];key=(r['id'],x0,y0,x1,y1)
        if key not in cache:
            sync();t0=time.perf_counter()
            with torch.no_grad(),models.amp():models.sam.set_image(read_image(root/r['image'])[y0:y1,x0:x1])
            cache[key]={k:[v.detach().float().cpu() for v in a] if isinstance(a,list) else a.detach().float().cpu() for k,a in models.sam._features.items()}
            if len(cache)>4:cache.popitem(last=False)
            sync();timing['feature_seconds']+=time.perf_counter()-t0
        cache.move_to_end(key)
        f={k:[v.to(device) for v in a] if isinstance(a,list) else a.to(device) for k,a in cache[key].items()}
        pts,labs,b=prompt(s,rng,mode,jitter);h,w=s['target'].shape
        if b is not None:pts=np.concatenate([b.reshape(2,2),pts]);labs=np.r_[2,3,labs]
        coords=pts/[w,h]*model.image_size
        with torch.no_grad():
            sparse,dense=model.sam_prompt_encoder(points=(torch.tensor(coords,dtype=torch.float32,device=device)[None],torch.tensor(labs,dtype=torch.int,device=device)[None]),boxes=None,masks=None)
        low,quality,_,_=model.sam_mask_decoder(image_embeddings=f['image_embed'],image_pe=model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse,dense_prompt_embeddings=dense,multimask_output=False,repeat_image=False,high_res_features=f['high_res_feats'])
        log=F.interpolate(low,size=(h,w),mode='bilinear',align_corners=False)
        target=torch.tensor(s['target'],dtype=torch.float32,device=device)[None,None]
        valid=torch.tensor(s['valid'],dtype=torch.float32,device=device)[None,None]
        return log,quality,target,valid
    def iou(log,t,v):
        m=log>0;truth=t>0;ok=v>0
        return ((m&truth)&ok).sum().float()/((m|truth)&ok).sum().clamp(min=1)
    def evaluate(samples,save=None):
        sync();ts=time.perf_counter();items=[]
        with torch.no_grad():
            for i,s in enumerate(samples):
                for mode in modes:
                    log,q,t,v=forward(s,mode,False)
                    items.append(dict(id=s['row']['id'],class_id=s['cls'],mode=mode,iou=float(iou(log,t,v))))
                    if save and mode==modes[0]:
                        np.savez_compressed(out/f'{save}_{i:04d}.npz',mask=(log[0,0]>0).cpu().numpy(),target=s['target'],valid=s['valid'],crop=s['crop'],image=s['row']['image'],class_id=s['cls'])
        # Equal weight per class, then per original image, then per prompt/component.
        scores={}
        for cls in classes:
            ids=sorted({r['id'] for r in items if r['class_id']==cls})
            if ids:scores[str(cls)]=float(np.mean([np.mean([r['iou'] for r in items if r['class_id']==cls and r['id']==id]) for id in ids]))
        sync();timing['eval_seconds']+=time.perf_counter()-ts
        return dict(macro_iou=float(np.mean(list(scores.values()))) if scores else None,per_class=scores,items=items)
    history=[];start=time.perf_counter();beststate=initial;beststep=0
    try:
        baseline=evaluate(sets['val']);best=baseline['macro_iou']
        test_before=evaluate(sets['test'],'test_before') if sets['test'] else None
        history.append(dict(step=0,val=baseline))
        pools={c:[s for s in sets['train'] if s['cls']==c] for c in classes}
        print('Trainable parameters:',sum(p.numel() for p in named.values()),flush=True)
        for step in range(1,steps+1):
            sync();ts=time.perf_counter()
            s=random.choice(pools[random.choice(classes)])
            log,q,t,v=forward(s,random.choice(modes),True);prob=log.sigmoid()
            bce=(F.binary_cross_entropy_with_logits(log,t,reduction='none')*v).sum()/v.sum().clamp(min=1)
            dice=1-(2*(prob*t*v).sum()+1)/((prob*v).sum()+(t*v).sum()+1)
            quality=F.mse_loss(q.reshape(-1),iou(log.detach(),t,v).detach().expand(q.numel()))
            loss=bce+dice+quality_weight*quality
            if not torch.isfinite(loss):raise RuntimeError('Non-finite loss')
            opt.zero_grad(set_to_none=True);loss.backward()
            assert all(p.grad is None for n,p in model.named_parameters() if n not in named)
            torch.nn.utils.clip_grad_norm_(list(named.values()),1.);opt.step()
            sync();timing['train_seconds']+=time.perf_counter()-ts
            if step%validate_every==0 or step==steps:
                ev=evaluate(sets['val']);history.append(dict(step=step,loss=float(loss.detach()),val=ev))
                print(step,'loss',float(loss.detach()),'val IoU',ev['macro_iou'],flush=True)
                if ev['macro_iou']>best:
                    best=ev['macro_iou'];beststep=step;beststate={n:p.detach().cpu().clone() for n,p in named.items()}
                (out/'history.json').write_text(json.dumps(history,indent=2))
        with torch.no_grad():
            for n,p in named.items():p.copy_(beststate[n].to(device))
        val_after=evaluate(sets['val'],'val_after')
        test_after=evaluate(sets['test'],'test_after') if sets['test'] else None
        metadata=dict(base='sam2.1_hiera_large',base_sha256=digest(models.root/'weights/sam2.1_hiera_large.pt'),class_ids=classes,
            class_id='multi_class_prompted_binary',context_pad_px=pad,steps=steps,best_step=beststep,lr=lr,seed=seed,
            ignore=ignore,min_area=min_area,prompt_modes=list(modes),quality_weight=quality_weight,
            trained='decoder single-mask path and IoU head; other modules frozen',trainable_names=list(named),
            trainable_parameters=sum(p.numel() for p in named.values()),baseline_val=baseline,after_val=val_after,
            test_before=test_before,test_after=test_after,data_manifest=rows,timing=timing,
            elapsed_seconds=time.perf_counter()-start,timing_note='train/eval include feature time; do not sum feature time again',
            torch_version=str(torch.__version__),device=str(device),gpu=torch.cuda.get_device_name() if str(device).startswith('cuda') else None,
            implementation='custom loop using official SAM modules, not official Trainer',
            limitation='No independent test evaluation' if not sets['test'] else 'Test split must remain independent; fixed GT-derived prompts')
        (out/'history.json').write_text(json.dumps(history,indent=2),encoding='utf8')
        torch.save(dict(parameters=beststate,metadata=metadata),out/'best_decoder.pt')
        (out/'metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf8')
        return metadata
    finally:
        with torch.no_grad():
            for n,p in named.items():p.copy_(initial[n].to(device))
        for p in model.parameters():p.requires_grad_(False)
        model.sam_mask_decoder.dynamic_multimask_via_stability=dynamic



## 4. 비교 조건
기본은 **동일 update step 수** 비교입니다. 이미지 수가 많아도 학습 시간/step 예산을 같게 두므로, 동일 epoch 비교는 아닙니다. 클래스 균등 샘플링은 복원 추출이며 한 번의 실험에서 모든 이미지가 실제 선택된다는 보장은 없습니다.

Train 이미지가 5/10장보다 적으면 해당 실험은 건너뛰고 이유를 출력합니다. 전체가 정확히 5 또는 10장이면 동일 부분집합 결과를 재사용합니다. 아래 실행 셀은 0장부터 순차 진행하며, 각 완료 결과를 즉시 파일에 저장합니다. 실행 셀을 다시 실행하면 새 비교 폴더를 만듭니다. 초기 점검에는 TRAIN_STEPS=2를 사용하세요.

In [ ]:
DEVICE = 'cuda'
TRAIN_STEPS = 200
LEARNING_RATE = 1e-5
VALIDATE_EVERY = 20
CONTEXT_PAD = 64
MIN_AREA = 8
SEED = 42
PROMPT_MODES = ('box_points','points','box')
DISPLAY_MASK_COUNT = 6
# GT 프롬프트를 사용하는 prompted segmentation 평가. GT는 test 업데이트에 사용되지 않습니다.


In [ ]:
"""Nested image-count plans. Validation/test untouched; class coverage explicit."""
import random

def plan_image_counts(rows, samples, classes, seed=42):
    train=sorted((r for r in rows if r['split']=='train'),key=lambda r:r['id'])
    if not train:raise ValueError('No train images')
    if not any(r['split']=='test' for r in rows):raise ValueError('Independent test required')
    coverage={r['id']:{s['cls'] for s in samples if s['row']['id']==r['id']} for r in train}
    empty=[id for id,c in coverage.items() if not c]
    if empty:raise ValueError('Train images without usable target masks: '+str(empty))
    # Cover each selected class first; shuffle remaining images with fixed seed.
    pool=train.copy();random.Random(seed).shuffle(pool);order=[];remaining=set(classes)
    while remaining:
        if not pool:raise ValueError('Selected class absent from training')
        chosen=max(pool,key=lambda r:len(coverage[r['id']]&remaining))
        if not coverage[chosen['id']]&remaining:raise ValueError('Selected class absent from training')
        order.append(chosen);pool.remove(chosen);remaining-=coverage[chosen['id']]
    required=len(order);order.extend(pool)
    plan=[dict(label='0',count=0,rows=rows,train_ids=[],baseline=True)]
    skipped=[];fixed=[r for r in rows if r['split']!='train']
    for n in (5,10):
        if len(train)<n:skipped.append(f'{n}: only {len(train)} train images');continue
        if n<required:skipped.append(f'{n}: class coverage requires {required} images');continue
        selected=order[:n]
        plan.append(dict(label=str(n),count=n,rows=selected+fixed,train_ids=[r['id'] for r in selected],baseline=False))
    # Keep the ALL label even if it matches 5 or 10; notebook reuses that result.
    plan.append(dict(label='all',count=len(train),rows=order+fixed,train_ids=[r['id'] for r in order],baseline=False))
    return plan,skipped


In [ ]:
samples=make_samples(DATA_ROOT,rows,TARGET_CLASSES,IGNORE_ID,CONTEXT_PAD,MIN_AREA)
plan,skipped=plan_image_counts(rows,samples,TARGET_CLASSES,SEED)
for split in ('train','val','test'):
    counts={c:sum(s['cls']==c and s['row']['split']==split for s in samples) for c in TARGET_CLASSES}
    print(split,counts)
    assert all(counts.values()), f'{split}에 선택 클래스의 유효 마스크가 없습니다'
for p in plan:print(p['label'],p['count'],'images',p['train_ids'])
for reason in skipped:print('SKIP:',reason)


## 5. 순차 실행: 각 학습 완료 직후 test 출력
표의 IoU는 클래스별·원본 이미지별 평균이며, 고정 프롬프트 방식들의 결과를 포함합니다. 각 이미지에는 첫 프롬프트 방식의 마스크를 표시합니다. 초록=예측, 자홍=GT. 시간은 모델 로딩을 제외한 각 학습 함수의 실행 시간입니다. feature 시간은 train/eval에 포함되므로 중복 합산하지 않습니다.

0장도 동일한 단일 마스크 경로와 crop·프롬프트를 사용합니다. 학습 전 test 재평가도 각 run에서 수행하여 기준선이 동일한지 확인합니다. 별도의 full image 자동 분할·DINO 검출 평가나 두께 오차 평가는 아닙니다.

In [ ]:
import csv, gc, uuid
from IPython.display import display, Markdown
BENCH_OUT = ROOT/'semantic_training_results'/('counts_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
BENCH_OUT.mkdir(parents=True,exist_ok=False)
(BENCH_OUT/'plan.json').write_text(json.dumps({'plan':plan,'skipped':skipped,'class_names':CLASS_NAMES,
    'steps_per_trained_run':TRAIN_STEPS,'prompt_modes':PROMPT_MODES,'selection_seed':SEED},ensure_ascii=False,indent=2),encoding='utf8')
summary=[];completed={};baseline_iou=None
def save_summary():
    (BENCH_OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf8')
    if summary:
        with (BENCH_OUT/'summary.csv').open('w',newline='',encoding='utf-8-sig') as f:
            writer=csv.DictWriter(f,fieldnames=list(summary[0]));writer.writeheader();writer.writerows(summary)
def show_result(run):
    display(Markdown('| 학습 이미지 | Test IoU | 기준선 대비 | Best step | 실행 시간(s) |\n|---|---:|---:|---:|---:|\n' +
        '\n'.join(f"| {r['label']} ({r['train_images']}) | {r['test_iou']:.4f} | {r['delta_pp']:+.2f}%p | {r['best_step']} | {r['seconds']:.1f} |" for r in summary)))
    files=sorted(run.glob('test_after_*.npz'))[:DISPLAY_MASK_COUNT]
    if files:
        fig,axes=plt.subplots(len(files),1,figsize=(9,3*len(files)),squeeze=False)
        for ax,p in zip(axes[:,0],files):
            with np.load(p) as z:
                x0,y0,x1,y1=z['crop'];im=read_image(Path(str(z['image'])))[y0:y1,x0:x1]
                ax.imshow(im)
                for name,color in [('target','magenta'),('mask','lime')]:
                    if z[name].any() and not z[name].all():ax.contour(z[name],levels=[.5],colors=[color])
                ax.set_title(f"class {int(z['class_id'])} / {Path(str(z['image'])).name}");ax.axis('off')
        plt.tight_layout();fig.savefig(run/'test_preview.png',dpi=150,bbox_inches='tight');plt.show();plt.close(fig)
for item in plan:
    key=tuple(item['train_ids'])
    if not item['baseline'] and key in completed:
        previous=completed[key]
        summary.append(dict(previous,label=item['label'],reused=True));save_summary()
        print('전체 집합이 이전 실험과 동일하여 결과 재사용:',previous['output'])
        show_result(Path(previous['output']));continue
    run=BENCH_OUT/('n_'+item['label'])
    display(Markdown(f"### 실행: {item['label']} / 학습 이미지 {item['count']}장"))
    trial_model=None
    try:
        # Never load an adapted checkpoint here: fresh official weights every time.
        trial_model=Models(ROOT,device=DEVICE)
        result=train_semantic(trial_model,DATA_ROOT,item['rows'],TARGET_CLASSES,run,
            steps=0 if item['baseline'] else TRAIN_STEPS,lr=LEARNING_RATE,pad=CONTEXT_PAD,
            min_area=MIN_AREA,ignore=IGNORE_ID,seed=SEED,validate_every=VALIDATE_EVERY,modes=PROMPT_MODES)
        score=result['test_after']['macro_iou']
        if item['baseline']:baseline_iou=score
        else:
            assert abs(result['test_before']['macro_iou']-baseline_iou)<1e-5, 'Baseline mismatch: inspect preprocessing/weights'
        entry=dict(label=item['label'],train_images=item['count'],test_iou=score,
            delta_pp=100*(score-baseline_iou),best_step=result['best_step'],seconds=result['elapsed_seconds'],
            output=str(run),reused=False)
        summary.append(entry);completed[key]=entry;save_summary()
        print('Class IoU:',result['test_after']['per_class'])
        show_result(run)
    except Exception as e:
        (BENCH_OUT/'failure.json').write_text(json.dumps({'run':item['label'],'error':str(e)},ensure_ascii=False,indent=2),encoding='utf8')
        print('이전 완료 결과는 보존되었습니다:',BENCH_OUT)
        raise
    finally:
        del trial_model;gc.collect()
        import torch
        if torch.cuda.is_available():torch.cuda.empty_cache()
print('Completed:',BENCH_OUT)


## 6. 전체 비교 그래프
Best step=0이면 validation 기준으로 학습 후 개선이 없어 초기 가중치를 선택한 결과입니다. 단 한 번의 부분집합 실험으로 데이터 수의 효과를 확정하지 말고, 필요하면 선택 seed를 바꿔 반복하세요. 여기서는 IoU를 계산하며 경계·두께 정확도는 별도 검증이 필요합니다.

In [ ]:
plt.figure(figsize=(8,4))
plt.plot([r['label'] for r in summary],[r['test_iou'] for r in summary],marker='o')
plt.xlabel('Training images (all = '+str(plan[-1]['count'])+')');plt.ylabel('Fixed test macro IoU');plt.ylim(0,1);plt.grid(alpha=.3)
plt.tight_layout();plt.savefig(BENCH_OUT/'learning_curve.png',dpi=160);plt.show()
print('CSV:',BENCH_OUT/'summary.csv')


## 검증 범위
노트북 문법·스키마, 중첩 부분집합 선택과 고정 test 유지 로직을 로컬 합성 데이터로 검사합니다. 실제 회사 데이터·GPU 학습은 미실행입니다. 이전 실제 Large 모델 CPU 검증은 시스템 RAM 부족으로 로딩 중 중단되었습니다. 독립 test 폴더와 모델 환경을 준비한 뒤 실행하세요.

## 체크포인트 저장 위치와 내용

각 실행 폴더 semantic_training_results/counts_<시각>_<ID>/n_0, n_5, n_10, n_all에 best_decoder.pt, metadata.json, history.json을 저장합니다. 전체가 5/10장과 같으면 해당 결과 폴더를 재사용하며 summary.csv의 output 열에 기록합니다.

best_decoder.pt에는 parameters(학습 대상으로 선택된 decoder 가중치)와 metadata(기본 SAM 가중치 SHA256, 클래스, 프롬프트, best step, 평가 결과 등)가 들어 있습니다. 원본 SAM2.1 Large 가중치와 함께 사용해야 합니다. optimizer/RNG 상태는 저장하지 않으므로 중단 step부터 정확히 이어가는 resume 체크포인트는 아닙니다. 현재는 해당 run의 최종 평가 완료 후 디스크에 저장되며, 실행 도중 중단하면 그 run의 best 가중치는 남지 않을 수 있습니다.

사용 전 기존 TEM_Offline_Metrology 패키지를 준비하고 ROOT를 지정하세요. 이 GitHub 노트북에 모델 가중치는 포함하지 않습니다. 실학습 GPU 검증은 미실행이며 문법·데이터·제어 흐름 검사만 완료했습니다.
